# Modelo baseline + metricas de evaluacion

Este notebook:
1. Escribe `src/models/baseline.py` (DenseNet121 preentrenado en NIH).
2. Escribe `src/evaluation/metrics.py` (AUC, accuracy, precision, sensibilidad, especificidad).
3. Corre smoke tests: carga el modelo, forward pass, y metricas sobre datos sinteticos.
4. Integra modelo + dataset + metricas en una pasada de validacion sobre val_ds.

**Runtime:** T4 GPU recomendado (el modelo preentrenado pesa ~28 MB, el forward es rapido pero mas rapido en GPU).

**Pre-requisitos:**
- Ya corriste `01b_fix_nih_dataset.ipynb` (nih.py funciona).
- `data/processed/train.csv`, `val.csv`, `test.csv` existen.


## 1. Setup: Drive, deps, paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR = f'{PROJECT_ROOT}/code'

import sys
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import torch
print(f'CUDA: {torch.cuda.is_available()}', end=' | ')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('CPU (funciona pero mas lento)')


In [ ]:
!pip install -q torchxrayvision scipy
print('deps OK')


## 2. Escribir `src/models/baseline.py`

In [ ]:
from pathlib import Path

content = r'''"""
src/models/baseline.py

Modelo baseline del Sprint 1: DenseNet121 preentrenado en NIH ChestX-ray14
(via torchxrayvision) adaptado a clasificacion binaria.

Justificacion:
  - TorchXRayVision provee DenseNet121 con pesos preentrenados en 14 patologias
    del NIH, incluyendo cardiomegalia. Usarlo como punto de partida es una
    ventaja sustancial frente a partir de ImageNet (las CXR son dominio muy
    distinto al de fotos naturales).
  - Reemplazamos la capa de clasificacion original (14 salidas, multi-label)
    por una nueva de 2 salidas (binary).

Dos modos de fine-tuning (controlados por cfg.model.finetune_mode):
  - "head": congelamos todo el backbone, entrenamos solo la capa final.
            Rapido, menos propenso a sobreajuste. Recomendado con pocos datos.
  - "full": entrenamos todo el modelo. Mayor flexibilidad, requiere mas datos.
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torchxrayvision as xrv


class BaselineCNN(nn.Module):
    """
    DenseNet121 de torchxrayvision con cabeza de clasificacion binaria.

    Args:
        backbone_weights: string del modelo preentrenado en xrv
                          (default: "densenet121-res224-nih").
        num_classes: numero de clases de salida (2 para binaria).
        finetune_mode: "head" (solo capa final) o "full" (todo el modelo).

    Input:
        x: tensor (batch_size, 1, 224, 224), rango [-1024, 1024].

    Output:
        logits: tensor (batch_size, num_classes). Aplicar softmax para probs.
    """

    def __init__(self, backbone_weights: str = "densenet121-res224-nih",
                 num_classes: int = 2, finetune_mode: str = "head"):
        super().__init__()

        if finetune_mode not in {"head", "full"}:
            raise ValueError(f"finetune_mode debe ser 'head' o 'full', "
                             f"no '{finetune_mode}'")
        self.finetune_mode = finetune_mode
        self.num_classes = num_classes

        # 1. Cargar DenseNet121 preentrenado de xrv
        self.backbone = xrv.models.DenseNet(weights=backbone_weights)

        # 2. Obtener dimension de features antes de la capa de clasificacion
        # El DenseNet121 de xrv tiene estructura:
        #   features -> ReLU -> AdaptiveAvgPool -> classifier
        # Vamos a quedarnos solo con la parte de features y agregar nuestra head.
        num_features = self.backbone.classifier.in_features  # 1024 para DenseNet121

        # 3. Reemplazar classifier original (14 clases multi-label) por uno binario
        self.backbone.classifier = nn.Linear(num_features, num_classes)

        # 4. Aplicar modo de fine-tuning
        self._apply_finetune_mode()

    def _apply_finetune_mode(self) -> None:
        """Congela/descongela parametros segun finetune_mode."""
        if self.finetune_mode == "head":
            # Congelar todo
            for param in self.backbone.parameters():
                param.requires_grad = False
            # Descongelar solo la capa final
            for param in self.backbone.classifier.parameters():
                param.requires_grad = True
        else:  # full
            for param in self.backbone.parameters():
                param.requires_grad = True

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass del modelo.

        IMPORTANTE: El DenseNet de xrv incluye una sigmoid activation al final
        por defecto (porque originalmente es multi-label). Usamos op_threshs=None
        implicitamente al no pasar ese parametro, pero de todas formas nuestro
        classifier reemplazado devuelve logits crudos.
        """
        # xrv.models.DenseNet.forward hace:
        #   features -> ReLU -> pooling -> classifier -> (opcional) sigmoid
        # Como reemplazamos el classifier, la salida son logits de 2 clases.
        # Evitamos la sigmoid final llamando los pasos internos manualmente:
        features = self.backbone.features(x)
        out = torch.relu(features, inplace=False)
        out = torch.nn.functional.adaptive_avg_pool2d(out, (1, 1))
        out = torch.flatten(out, 1)
        logits = self.backbone.classifier(out)
        return logits

    def trainable_params(self) -> int:
        """Cuenta parametros entrenables (util para logging)."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def total_params(self) -> int:
        """Cuenta parametros totales."""
        return sum(p.numel() for p in self.parameters())


# ---------------------------------------------------------------------------
# Factory
# ---------------------------------------------------------------------------
def build_baseline_model(cfg) -> BaselineCNN:
    """
    Construye el modelo baseline desde la config.

    Args:
        cfg: objeto Config cargado con load_config().

    Returns:
        BaselineCNN listo para entrenar o evaluar.
    """
    model = BaselineCNN(
        backbone_weights=cfg.model['backbone'],
        num_classes=cfg.model['num_classes'],
        finetune_mode=cfg.model['finetune_mode'],
    )
    return model


# ---------------------------------------------------------------------------
# Smoke test
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    import sys
    from pathlib import Path

    code_dir = Path(__file__).resolve().parent.parent.parent
    if str(code_dir) not in sys.path:
        sys.path.insert(0, str(code_dir))

    from src.training.utils import load_config, get_device

    cfg = load_config(code_dir / "configs" / "baseline.yaml")
    device = get_device()

    model = build_baseline_model(cfg).to(device)
    print(f"Modelo construido: {model.__class__.__name__}")
    print(f"  backbone: {cfg.model['backbone']}")
    print(f"  num_classes: {cfg.model['num_classes']}")
    print(f"  finetune_mode: {cfg.model['finetune_mode']}")
    print(f"  parametros totales:     {model.total_params():>12,}")
    print(f"  parametros entrenables: {model.trainable_params():>12,}")

    # Forward con un batch dummy
    x = torch.randn(4, 1, 224, 224, device=device) * 500  # rango tipo xrv
    logits = model(x)
    print(f"\nForward OK:")
    print(f"  input shape:  {x.shape}")
    print(f"  output shape: {logits.shape}")
    print(f"  output range: [{logits.min():.3f}, {logits.max():.3f}]")

    # Probabilidades via softmax
    probs = torch.softmax(logits, dim=1)
    print(f"  probs shape:  {probs.shape}")
    print(f"  probs sum:    {probs.sum(dim=1).tolist()}  (deben ser ~1.0)")
'''
path = f'{CODE_DIR}/src/models/baseline.py'
Path(path).parent.mkdir(parents=True, exist_ok=True)
Path(path).write_text(content, encoding='utf-8')
print(f'OK -> {path} ({len(content):,} chars)')


## 3. Escribir `src/evaluation/metrics.py`

In [ ]:
content = r'''"""
src/evaluation/metrics.py

Metricas de evaluacion para clasificacion binaria de CXR.

Metricas implementadas (alineadas con el Project Charter):
  - AUC (area bajo la curva ROC)
  - Accuracy
  - Precision (VP / (VP + FP))
  - Sensitivity / Recall / True Positive Rate (VP / (VP + FN))
  - Specificity / True Negative Rate (VN / (VN + FP))

Convenciones:
  - Clase positiva: label 1 (Cardiomegaly)
  - Clase negativa: label 0 (No Finding)
  - AUC se calcula sobre probabilidades (no sobre predicciones binarias).
  - Las demas metricas requieren umbralizar las probabilidades (default 0.5).
"""
from __future__ import annotations

from dataclasses import dataclass, asdict

import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    roc_auc_score,
)


# ---------------------------------------------------------------------------
# Estructura de resultados
# ---------------------------------------------------------------------------
@dataclass
class ClassificationMetrics:
    """Contenedor tipado para las 5 metricas + conteos de confusion matrix."""
    auc: float
    accuracy: float
    precision: float
    sensitivity: float  # = recall = TPR
    specificity: float  # = TNR
    # Conteos de confusion matrix (utiles para debug)
    tp: int
    fp: int
    tn: int
    fn: int
    # Metadata
    threshold: float
    n_samples: int

    def as_dict(self) -> dict:
        return asdict(self)

    def as_short_dict(self) -> dict:
        """Solo las 5 metricas principales, en porcentaje."""
        return {
            'auc': self.auc,
            'accuracy': self.accuracy,
            'precision': self.precision,
            'sensitivity': self.sensitivity,
            'specificity': self.specificity,
        }

    def __str__(self) -> str:
        return (
            f"AUC={self.auc:.4f} | "
            f"Acc={self.accuracy:.4f} | "
            f"Prec={self.precision:.4f} | "
            f"Sens={self.sensitivity:.4f} | "
            f"Spec={self.specificity:.4f} "
            f"(TP={self.tp} FP={self.fp} TN={self.tn} FN={self.fn})"
        )


# ---------------------------------------------------------------------------
# Calculo de metricas
# ---------------------------------------------------------------------------
def compute_metrics(y_true: np.ndarray | torch.Tensor,
                    y_prob: np.ndarray | torch.Tensor,
                    threshold: float = 0.5) -> ClassificationMetrics:
    """
    Calcula las 5 metricas para clasificacion binaria.

    Args:
        y_true: labels verdaderos (0 o 1), shape (N,).
        y_prob: probabilidad de la clase positiva, shape (N,) en [0, 1].
        threshold: umbral para binarizar probabilidades (default 0.5).

    Returns:
        ClassificationMetrics con las 5 metricas y conteos.
    """
    y_true = _to_numpy(y_true).astype(int)
    y_prob = _to_numpy(y_prob).astype(float)

    if y_true.shape != y_prob.shape:
        raise ValueError(f"Shapes distintos: y_true={y_true.shape}, "
                         f"y_prob={y_prob.shape}")

    # Binarizar con el umbral
    y_pred = (y_prob >= threshold).astype(int)

    # AUC requiere al menos ambas clases presentes
    if len(np.unique(y_true)) < 2:
        # Edge case: todos los labels son iguales (raro pero posible en batches).
        auc = float('nan')
    else:
        auc = roc_auc_score(y_true, y_prob)

    accuracy = accuracy_score(y_true, y_pred)

    # Confusion matrix. Fijamos labels=[0, 1] para garantizar shape 2x2
    # incluso si el batch no tiene ambas clases.
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    # Precision: VP / (VP + FP)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0

    # Sensitivity / Recall: VP / (VP + FN)
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    # Specificity: VN / (VN + FP)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return ClassificationMetrics(
        auc=float(auc),
        accuracy=float(accuracy),
        precision=float(precision),
        sensitivity=float(sensitivity),
        specificity=float(specificity),
        tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn),
        threshold=float(threshold),
        n_samples=int(len(y_true)),
    )


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def _to_numpy(x) -> np.ndarray:
    """Convierte torch.Tensor o np.ndarray a np.ndarray."""
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def logits_to_probs(logits: torch.Tensor, positive_class: int = 1
                    ) -> torch.Tensor:
    """
    Convierte logits (B, num_classes) a probabilidad de la clase positiva (B,).

    Default positive_class=1 (Cardiomegaly en nuestra convencion).
    """
    probs = torch.softmax(logits, dim=1)
    return probs[:, positive_class]


# ---------------------------------------------------------------------------
# Smoke test
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    # Caso sintetico: 100 muestras, 30% positivas, modelo razonablemente bueno
    np.random.seed(42)
    n = 100
    y_true = np.random.binomial(1, 0.3, size=n)
    # Probabilidades correlacionadas con y_true (modelo con 85% de acierto aprox)
    y_prob = np.where(y_true == 1,
                      np.random.beta(5, 2, size=n),      # positivos -> probs altas
                      np.random.beta(2, 5, size=n))      # negativos -> probs bajas

    metrics = compute_metrics(y_true, y_prob, threshold=0.5)
    print("Smoke test metrics.py:")
    print(metrics)
    print()
    print("As dict:", metrics.as_short_dict())
'''
path = f'{CODE_DIR}/src/evaluation/metrics.py'
Path(path).parent.mkdir(parents=True, exist_ok=True)
Path(path).write_text(content, encoding='utf-8')
print(f'OK -> {path} ({len(content):,} chars)')


## 4. Smoke test del modelo

Cargamos el modelo preentrenado y hacemos un forward con un batch dummy.
**La primera vez toma ~30s** porque descarga los pesos (~28 MB) de torchxrayvision.

In [ ]:
# Limpiar cache de imports
for mod in list(sys.modules.keys()):
    if mod.startswith('src.'):
        del sys.modules[mod]

from src.training.utils import load_config, get_device
from src.models.baseline import build_baseline_model

cfg = load_config(f'{CODE_DIR}/configs/baseline.yaml')
device = get_device()

print('Construyendo modelo (descargar pesos si es la primera vez)...')
model = build_baseline_model(cfg).to(device)
model.eval()

print(f"Modo: {cfg.model['finetune_mode']}")
print(f'Parametros totales:     {model.total_params():>12,}')
print(f'Parametros entrenables: {model.trainable_params():>12,}')

# Forward dummy
x = torch.randn(4, 1, 224, 224, device=device) * 500
with torch.no_grad():
    logits = model(x)
print(f'\nForward OK:')
print(f'  input shape:  {x.shape}')
print(f'  output shape: {logits.shape}')
print(f'  logits range: [{logits.min():.3f}, {logits.max():.3f}]')

probs = torch.softmax(logits, dim=1)
print(f'  probs shape:  {probs.shape}')
print(f'  probs[0]:     {probs[0].tolist()}  (suma = {probs[0].sum().item():.3f})')


## 5. Smoke test de metricas (datos sinteticos)

In [ ]:
from src.evaluation.metrics import compute_metrics
import numpy as np

np.random.seed(42)
n = 100
y_true = np.random.binomial(1, 0.3, size=n)
# Modelo sintetico razonablemente bueno
y_prob = np.where(y_true == 1,
                  np.random.beta(5, 2, size=n),
                  np.random.beta(2, 5, size=n))

metrics = compute_metrics(y_true, y_prob, threshold=0.5)
print(f'Muestras: {metrics.n_samples}')
print(metrics)


## 6. Integracion: evaluacion del modelo SIN entrenar sobre val_ds

Este es el smoke test mas importante. Corre el modelo preentrenado sobre val
(antes de reentrenarlo con nuestros datos) para ver:
  1. Si todo el pipeline corre end-to-end.
  2. Como se desempena el modelo "tal cual sale de la caja".

Expectativa: como el modelo de xrv viene preentrenado en NIH con Cardiomegaly
como una de las 14 patologias, **podria dar AUC decente directamente** (>0.70).
Pero ojo: el classifier original fue REEMPLAZADO por uno nuevo con pesos
aleatorios, asi que en realidad el modelo aun no ha sido entrenado para nuestra
tarea binaria. Las metricas iniciales seran cercanas al azar (~50%).

Eso esta bien: valida que el pipeline corre, y establece la "linea base random".

In [ ]:
from torch.utils.data import DataLoader
from src.datasets.nih import build_nih_datasets
from src.datasets.transforms import build_train_transform, build_eval_transform
from src.evaluation.metrics import compute_metrics, logits_to_probs

eval_tfm = build_eval_transform(image_size=cfg.preprocessing['image_size'])
train_tfm = build_train_transform(image_size=cfg.preprocessing['image_size'])

_, val_ds, test_ds = build_nih_datasets(cfg, train_tfm, eval_tfm)

val_loader = DataLoader(val_ds, batch_size=cfg.training['batch_size'],
                        shuffle=False, num_workers=0)

# Forward pass sobre todo val
all_probs = []
all_labels = []

model.eval()
with torch.no_grad():
    for batch in val_loader:
        images = batch['image'].to(device)
        labels = batch['label']

        logits = model(images)
        probs = logits_to_probs(logits, positive_class=1)

        all_probs.append(probs.cpu())
        all_labels.append(labels)

all_probs = torch.cat(all_probs).numpy()
all_labels = torch.cat(all_labels).numpy()

print(f'Muestras evaluadas: {len(all_labels)}')
print(f'Positivos verdaderos: {all_labels.sum()}')
print(f'Probabilidad media predicha: {all_probs.mean():.3f}')
print()

metrics_init = compute_metrics(all_labels, all_probs, threshold=0.5)
print('Metricas del modelo SIN entrenar (classifier head con pesos aleatorios):')
print(metrics_init)
print()
print('Nota: estas metricas son la LINEA DE PARTIDA. Esperamos que despues')
print('de entrenar suban significativamente.')


## 7. Siguiente paso

Si todo corre sin errores:
- Pipeline de datos: **OK**
- Modelo baseline: **OK**
- Metricas: **OK**
- Integracion modelo+datos+metricas: **OK**

**Falta solo el loop de entrenamiento.** Vuelve al chat para recibir:

1. `src/training/train.py` - loop de entrenamiento con validacion y early stopping.
2. `scripts/run_train.py` - entrypoint que carga config, llama al loop, guarda checkpoint.

Con esos dos, harás tu primer entrenamiento real.
